# ```Imports```

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
# Torch will handle all the established maths behind the curtain fro us, lot of calculus and tensors.
# let's set up owr devices now;
device='cuda' if torch.cuda.is_available() else 'cpu'

# hyper parameters
block_size=8
batch_size=4
# The avobe parameters are very very important for training.
# We will see how important they are when we scale our model and the parameters will keep getting more and more complex.
max_iters=1000
# evaluate_interval=2500
learning_rate=3e-4 # 3^(-4)
# We will have to experiment with it, for best results
# Very slow learning rate= 0.0001
eval_iters=250
#dropout=0.2 # It's gonna dropout random neurons but it is disabled in the eval mode
# It's gonna help the model learn better, meaning when there are noises and when the things arent quiet in the right place
# We don't want some neuron to get priority and making all of the weird decisions.

# SideEffects
print(device)
# cuda is available
#!nvidia-smi

# ```Dataset```
Let's download a small dataset from ```Project Guttenberg```(https://www.gutenberg.org)
We are going to train a ```transformer``` or a ```bigram language model``` with our preparred dataset

In [ ]:
# We are opening the text in read mode with encoding 'utf-8'
with open('wizard_of_oz.txt','r',encoding='utf-8') as f:
    text=f.read()
# Plain text is a list of strings or char s
print(len(text))
chars=sorted(set(text))
# Sorted is not for sorting but for creating a list from a set
vocab_size=len(chars)
print(vocab_size)
print(chars)

In [ ]:
# Now we are are going to use tokenizers with generative for loops and custom functions;
string_to_int={ch:i for i,ch in enumerate(chars)}
int_to_string={i:ch for i,ch in enumerate(chars)}
encode=lambda s:[string_to_int[c] for c in s]
#decode=lambda l:''.join([chars[int(*j)] for j in l])
decode=lambda l:''.join([int_to_string[j] for j in l])
# All the errors because I had an extra pair of []s

In [ ]:
# same as the above lambda
def encode(s):
    encode_list=[]
    for c in s:
        encode_list.append(string_to_int[c])
    return encode_list
def decode(l):
    decode_list=[]
    for j in l:
        decode_list.append(int_to_string[j])
    decoded_text=''.join(decode_list)
    return decoded_text

In [ ]:
# Let's test the lambdas
encoded_hello=encode('hello')
# So now we have a conversion from characters to integers so let's decode it back into our feature
decoded_hello=decode(encoded_hello)
encoded_hello,decoded_hello
# Is this the so called embedding?

# ```Tokenizers```:
Right now we have been using ```character level``` tokenizer, it takes each ```character``` and converts them into an ```integer```.
We have a ```small vocabulary``` but a ```large number of tokens```(232308 characters) that we have converted into ints.
If we had a ```word level tokenizer``` meaning all the possible ```meaningful arrangements``` of group of characters.
So the ```biggest real world word``` having ```r number of characters``` will belong to ```{{nPr+np(r-1)+.....+nP(r-r)}and{all the words in the dictionary}}``` while n=80 according to above len('tf-8' encoding)
So Ton of more tokens, maybe millions or billions of tokens
Ususally we will be ```between``` character level or word level tokenizer
# ```Efficiency``` in Dataset
In the context of language model it is very important that we are efficient with our data.
Just having a giant stream might not work the best

In [ ]:
# let's unleash the torch! khuuuuuuu
# Our guy wants to use torch.long so int64; he thinks it's because we will be dealing with a super long sequence of tensors, so bigger chunks in ram assignment the better
data=torch.tensor(encode(text),dtype=torch.long)
data.ndim,data[:100]
# It has only one dimensions as bunch of integers inside one set of []s

# Let's talk about training and validation ```splits```
We need to ```train``` on a part of the ```corpus``` and keep remaining parts for ```validation``` or testing
If we train on the ```whole corpus```, it will ```memorize``` everything; that spoils the intelligence.
That testing part is what makes the ```adaptive nodes``` inside the black box

In [ ]:
# Let's get the split at around 80% of the data and n gives us exact iteror index at the exact slice of 0.8*232308 from the start
n=int(0.8*len(data))
train_data=data[:n] # 80% of the data
val_data=data[n:] # remaing 20% of the data
# So the train_data consists of the inputs and val_data consists of the y_i s?

def get_batch(split):
    # Reassigning data;
    data=train_data if split=='train'else val_data # Is train some kind of flag?
    # Two exclusive stream of data selection
    ix=torch.randint(len(data)-block_size,(batch_size,)) # The second argument takes in a touple and we have a V so 4D vector, for stacking 8 members in each of the 4 rows later
    # len(data)-block_size is the random range of torch.randint() formation, meaning each elements ranging from this huge number.
    # We are taking the entirity of either train_data or val_data minus the block size so that overlapping doesn't happen as we are already dealing with some batch of 8 chars somewhere in data
    # So ix is a 4D vector of huge nundom numbers!
    #[data[i:i+block_size] for i in ix] # Just takiing horizontal slices of block size from the indices of encoding from ix in each iteration
    # data[x:y] refers (from either train_data or val_data) consecutive members from x to y excluding y, so now let's torch.stack the 8 members of the above list comprehension
    #print(ix)
    inputs=torch.stack([data[i:i+block_size] for i in ix])
    # Now we will slide by one char so;
    targets=torch.stack([data[i+1:i+block_size+1] for i in ix])
    # One more thing, let's send the inputs and outputs to device
    inputs,targets=inputs.to(device),targets.to(device) # WE ARE USING device='cuda:0' for these tensors!!
    # These batches will not be processed sequentially on cpy rather parallelly on gpu!
    # That's how we push any piece of data into gpu!
    return inputs,targets
    

#x,y=get_batch('train')
# Train is just a flag. We gave flag that we are in the training zone!
#x,y

# ```Bigram Bi Prefix Means 2```
like the ```in between``` aproach in animation, but different in a sense that if it has a character h then an e after that will have ```bigger probabilty``` than l given the  real world usecase of ```meaningful word making```.
As ```two letters are involved``` and thus ```weight```(of the second letter) is determined based on the ```position of two letters one after another``` it's called a bigram model.
Similar models we can generate for our ```steps lesson from mewstro``` where the second note(```target```) is weighted by the context of the first note(```context```).

# ```Block Size```
Instead of taking the whole corpus size we are going to take ```small snippets```.
We are going to ```make some predictions``` and we are going to ```decide some targets``` out of that.
Let's take a ```block size``` of 5(probably a trade off between the compression and the accuracy of the predictions) and observe two ```consecutive interlocking blocks``` in a row.

1. [...[5, 67, 21, 58, 40],35 ...]
2. [... 5,[67, 21, 58, 40, 35]...]

So the next block is ```slided over``` the other at the ```interlocked zone``` of a text_list==[67, 21, 58, 40]

In [ ]:
# To implement the blocks pythonically we will start the snippet from the slice where the first_interlocking_snippet==encoded_text[:block_size] and second_interlocking_snippet==encoded_text[1:block_size+1] 
#block_size=8
x=train_data[:block_size] # x is a list
y=train_data[1:block_size+1] # y is a list
# Both x,y are the interlocking slices from the training data which starts from data[0]
for t in range(block_size): # iteration number = block_size && t=0, t+=1 on each iteration upto t=block_size-1
    # Context is the elements inside the list;
    context= x[:t+1] # assignment of context, meaning the first of the two interlocking (list|chunk|block)-populating like the sheeps from x[0:1] to x[0:block_size]
    # target is the element outside the list but the index=0 of the second (list|chunk|block)
    target=y[t]
    # We are not appending so all the assignments are being reassignments withe modified indexes on each iteration
    print(f'context=={context} gives target=={target}')

# This Isn't Scalable Yet!
This is ```sequencial```; sequencial is another way to describe ```what a cpu does as opposed to gpus```.
CPUs can do a lot of complex tasks very quickly but it only happens sequencially.
On the other hands, GPUs we can do a lot of simplersmaller computations very very quickly and in parallel.
The processors of gpus are much inferierior as units but they come in large numbers.
So what we can do is we can take each of these little blocks and then we can stack them to push these to GPUs to scale our training, a lot.

# Batches as a Hyper Parameter
For illustration let's take 4 blocks and stack them on top of one another;
```
[... ...]
[... ...]
[... ...]
[... ...]
```
We can represent this new block of 4 blocks as a new hyper parameter called batch size.
It tells us how many of these sequences or block of loops we can process in parallel. (So a 2d tensor?)


```
batch_size=[[... ...]
            [... ...]
            [... ...]
            [... ...]]
```                   

So we just went over how we get batches or rather how we can utilize this second layer of representation to accelerate the training process using the parallel processing muscle of the GPUs.
Let's take our block_size variable up at the import cell where other global variables like batch_size will be defined.


In [ ]:
@torch.no_grad()
# it makes sure that pytorch doesn't add gradient accumulations here, that would be computionally heavy
# Get batch is not using the nn and everything comes with it, but estimate_loss() does
# We need to make sure it's not using the gradient heavy lifting due to torch.optim.AdamW()
def estimate_loss(): # No inputs? May be model
    out={}
    # We kept the evaluation at top because we are testing the model here
    # We wanna see how well it does with given set of data from get batch.
    model.eval() # Turns on evaluation mode on the model, testing
    # dropout and batch_normalization behaves differently in this mode.
    # What we are doing is we are using the entire network, we need everything to be working sort of together
    # And we wanna see how well does it actually perform.
    # Training mode is just sampling, experimenting, we are doing weird staffs, sort of challenging the model. 
    # And in evaluating mode we wann see the model in its optimal form and we wanna see how good of results it actually produces.
    for split in ['train','val']: # So two iterations and will create different models based on different input,output batches tensors 
        # Here we get the background of split lol
        losses=torch.zeros(eval_iters) # getting tensor with 250 zeros
        for k in range(eval_iters): # 250 iterations
            X,Y=get_batch(split) # First train then val, and capital X and Y
            logits,loss=model(X,Y) # ceated another model with X,Y
            losses[k]=loss.item() # loss is a 0D tensor, so we can unwrap the it into a scalar and assign it to losses[k]th zero
        out[split]=losses.mean() # It just creates a dictionary of one 'train':{losses.mean()} and one 'val':{loses.mean()}    
    model.train() # turns on training mode on the model
    '''We are here to estimate loss and evaluate the model, so training is not being cattied on in these loop blocks'''
    '''We are just sending the model to train after the evaluation is complete and just before returning the out dictionary'''
    return out # Completed 250 iteration for out['train'] and 250 for out['val']
            

# Time For Neural Network
# "`logits`"
It is a blend (portmanteau) of "logistic unit," coined by statistician Joseph Berkson in 1944, and in deep learning it refers to the raw, unnormalized score vector produced by the final linear layer of a neural network before applying an activation function like Softmax or Sigmoid.
- The raw, unnormalized output coming straight out of your final nn.Linear layer. These numbers can be any real value (negative, zero, or very large positive numbers). $$z = Wx + b\rightarrow(Logits)$$ 
# Why SHAPE and VIEW Methods
According to the documentation the method expects a shape as $(C),(N,C)or(N,C,d_1,d_2,...,d_K)$  with $K\ge 1$ where;
$$C=\text{number of classes}$$
$$N=\text{batch size}$$
- For our case batch size is denoted by B So we need to redistribute our acquired shape (B,T,C) from nn.Embedding() into (B,C) for logits and (C) for targets. We could just retype it by hand but .view() gives us this scope of reallocation of dimensions.
- Also for our case we are implementing the T time by multiplying it with B to get a more efficient version of B and for targets we are implementing the Batch size B by multiplying it with C to get a more efficient version of C.
## `SHAPE And VIEW Methods Help Reshapeing The Existing Tensors`
## Loss
Let's say we have 80 chars in our vocabulary, and we start our model with random weights and biases, meaning the first forward feed. In this state we have a $\frac{1}{80}$ chance to predict the next token correctly and to translate $-ln(\frac{1}{80})$% into is what loss actually looks like.
So pretty much the point is to minimize the loss.
While updating $\vec{\theta}$ according to $\vec{\theta}'=\vec{\theta}-\alpha\vec{\nabla_{w,b}C}$ the $\alpha$, the so called learning rate, is what enables us to have a proper float value to subtract|add with the analogous float value of $\vec{\theta}$.
We will keep adjusting the parameters so that our gradients change it's course towards the minima on each iteration.
Gradient descent optimizer.
Let's browse through torch.optim to explore other optimizers.
# `Cross Entropy`
Cross-entropy is a loss function from information theory that measures the difference between two probability distributions—specifically, how surprised your model's predicted probabilities (q) are by the true ground-truth labels (p).
1. What Cross-Entropy Actually Is (The Core Formula)
Given the true distribution p and your model's predicted distribution q across C classes:
$$H(p,q)=-\sum _{i=1}^{C}p_{i}\log (q_{i})$$
Because ground-truth classification targets are one-hot (where p_y = 1 for the single true class y and 0 for all other classes), every term in the sum disappears except the true class y:
$$\text{Loss}=-\log (q_{y})$$
In PyTorch: F.cross_entropy(logits, targets) takes raw logits (z) directly, applies log_softmax internally for numerical stability, and picks -log(q_y).
2. Different kinds of `Cross-Entropy`={1:' Multi-Class Cross-Entropy',2:'Binary Cross-Entropy (BCE) with Logits',3:'Plain Binary Cross-Entropy (BCE)',4:'Negative Log-Likelihood (NLL)',5:'Weighted Cross-Entropy',6:'Label-Smoothed Cross-Entropy',7:'Focal Loss',8:'KL Divergence (Relative Entropy)'}
# Breaking Down the `logits[:,-1,:]` Range In Each Dimension
The variable `logits` usually represents a 3D tensor with the shape `(batch_size, sequence_length, vocabulary_size)`. Here is how the slice `[:, -1, :]` targets each dimension:

- `:` (First Dimension - Batch Size): The colon selects all batches in the current forward pass.
- `-1` (Second Dimension - Sequence Length): Python uses negative indexing to count from the back. `-1` selects only the `i=-1` (the most recently generated or inputted token) in the text sequence. 
- `:` (Third Dimension - Vocabulary Size): The final colon retains the entire vocabulary distribution (raw scores for every possible word/token) for that last position.

If We had [:,:,:] it would select all. For each in betweem comma we can apply the `iterable[j:k]` aproach meaning the logits being a 3D tensor can show 3 separate indexing option like `logits[i:j,k:l,m:n]`.
# "`probs`"
What you get after passing z through Sigmoid (for binary classification) or Softmax (for multi-class / next-word prediction), squashing those raw numbers into probabilities between 0 and 1 that add up to 1.
$$output,\vec{a}^l= σ(z)\text{ or }Softmax(z)\rightarrow(Probabilities)$$
# Softmax `Activation Function`
Softmax is a mathematical activation function that converts a vector of raw, unbounded numbers (logits, z) into a clean probability distribution where every number is between 0 and 1 and all numbers add up to 1.0.
Given a vector of raw logits $z = [z_1, z_2, ..., z_K]$:
$$\text{Softmax}(z_{i})=\frac{e^{z_{i}}}{\sum _{j=1}^{K}e^{z_{j}}}$$
1. Exponentiate $e^{z_i}$: Turns every number positive (even negative logits like -5.0 become small positive numbers $e^{-5} ≈ 0.0067$) and amplifies the largest numbers.
2. Normalize (`÷ sum`): Divides each exponential by the sum of all exponentials so the entire vector adds up to `1.0` (100%).
## For PyTorch Usage (`dim is crucial!`)
- Always pass $dim=-1$ to `normalize` across the `last axis (the 80 classes)`!

`Different kinds of softmax`={1:'Standard Softmax',2:'Log-Softmax',3:'Softmax with Temperature (T)',4:'Softmin',5:'Gumbel-Softmax',6:'Sparsemax / Entmax',7:'Hierarchical / Adaptive Softmax',8:'Online / FlashAttention Softmax'}

In [ ]:
class BigramLanguageModel(nn.Module):
    # It will be a subclass of nn.Module and inherit the superclass variables and methods
    # If we use nn.Linear inside this class that will mean the nn.Linear paeameters are learnable; the weight matrix can be changed with gradient descent.
    def __init__(self,vocab_size):
        super().__init__() # All the nn init parameters are loaded here
        self.token_embedding_table=nn.Embedding(vocab_size,vocab_size) # So just 80 from 80; comparing with the gpt example of using_cuda.ipynb
        # It's just creating an attribute .token_embedding_table where some attributes are from super class

    def forward(self,index,targets=None): # So we can exclude targets and everything will stay just fine
        # This is a class method where we define everythin feed forward.
        # We are making custom function instead of using what it has by default because it is a good practice to know what is happening behind the scene in our model.
        # We wanna know exactly what's going on so that we can be more flexible and creative.
        # If we are using weird data and defaults are not used to it we are going to get bugs from that. 
        logits=self.token_embedding_table(index) # Pulling the attribute from the init block which is just a nn.Embedding() ready and hungry for indices.
        '''Our logits is 3D, having size=(index,vocab_size,vocab_size'''
        '''Before the above assignment we had an index list input from get_batch('train), and now we have a 3d'''
        # The nn.Embedding() is pretty much just a look up table, we are gonna have a giant sort of grid of what the predictions are gonna look like.
        if targets is None:
            '''First pass doesn't have targets, so targets is none. We do not go to else block to get dimension error in def generate() block during calling <logits=logits[:,-1,:]>'''
            loss=None
            '''Therefore the loss is also none'''
        else:
            '''not our first attempt'''
            '''Coming from the first optimizer block iteration'''
            # We are gonna use a function called .view which is gonna help us sorta reshape how logit looks like in human form.
            # B -> batch; T - Time Dimension, through the generation for some we do know and some we don't know yet; C-> Channel stands for vocab_size 
            B,T,C=logits.shape
            '''It's stacking unique 80 vocab_size numbered v^Ts, the nn.Embedding(.weight.detach().numpy())[index] lists from the nn.Embedding().weights.detach() tensor of size (80,80)'''
            # Ar above, we just assined the touple from logit.shape into B,Tand C
            # So basically logits will have a table of index amount of rows and vocab_size amount of columns.
            # The table has size=(index,vocab_size).
            # Below we are reducing logit.shape from 3D to 2D;
            logits=logits.view(B*T,C) # B and T are usually blended together because they are not as important here
            '''It had a plan of length 8 points, width 4 points and height 80 point. Now it's rhino unwrapped in a line of 32 points and 80 points high'''
            # As long as the logits and the targets have same (B,T) we shall be fine.
            # Let's now initialize our targets, it's just 1D;
            targets=targets.view(B*T) # Probably targets are supposed to have one dimension of elements less.           
            '''targets was already 2D the same size as index, and here we are unwrapping it into a vector of 32 rows.'''
            '''The C=80, vocab size = unique chars is absent here because we did not assign nn.Embedding() look up table's 80 points high columns'''
            # Why do we have to reshape the logits and targets? well the answere falls into what does cross_entropy() expects the shapes to be of its arguments.
            loss=F.cross_entropy(logits,targets) # We are bringining functional's cross_entropy() method which takes in two args.
            '''The so called vector nabla C'''
            '''it's a 0D tensor, a scalar, so loss.item works'''
            '''
            loss = F.cross_entropy(
                input,              # Raw logits (z) of shape (batch_size, num_classes)
                target,             # Ground-truth class IDs of shape (batch_size,) as LongTensor
                weight=None,        # Optional class weights for imbalanced datasets
                ignore_index=-100,  # Target ID to skip (e.g., your padding_idx!)
                reduction='mean'    # 'mean', 'sum', or 'none'
            )
            '''
            # We provided the cross_entropy() with logits and targets, two .view() methods of our nn.Embedding() objects. 
            # It is very important to stay well informed with what and how each pytorch function expects the arguments to be.
        return logits,loss
        # Logits are a kind of probability distribution, a more sophisticated version of normalization; it's kinda the probability distribution of all  the 80 other letters in the context of one letter.

    # Now Let's define generate which returns index;
    def generate(self,index,max_new_tokens):
        '''This is not included in the forward pass'''
        # index is the (B,T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions; probably index will be a class variable, we are creating everything out of indexes. WHAT IS INDEXES!!!!!
            logits,_=self.forward(index) # logits and losses are being generated exactly as defined in forward() block
            '''We have received all our 4 delta vectors from 4 layers of neurons'''
            # Focus only on the last time step
            logits=logits[:,-1,:] # Becomes (B,C)
            '''Getting last indexes from logits[1][-1] and logits[2][-1] carrying the last overlapping slices [5, 67, 21, 58, 40] and [67, 21, 58, 40, 35]'''
            # Apply functional's softmax to get probabilities. Goal: softer nudges on our random logits            
            '''The softmax activation function after cross entropy is what gives us inverse acrivation over the loss which then upon applied nabla gets subtracted from the stored a^l s'''
            probs=F.softmax(logits,dim=-1) # (B,C) to (B,1), column vector as activations
            '''softmax is a normalization commonly used in the output layer; sigmoid for the output layer'''
            '''If we had bunch of nn.Linear() s in a row, they will all get sqweezed together and be applied with one transformation that summs up altogether'''
            '''It is computationally much less expensive'''
            # Sample from the distribution, again index! next_index!
            index_next=torch.multinomial(probs, num_samples=1) # Plugging in (B,1)
            '''We got it by updating our weights and biases'''
            # Append sampled index to the running sequence, I KNOW IT, IT MAKES LONGER TENSORS CONCATINATING SMALLER ONES LIKE LISTS!
            index=torch.cat((index,index_next),dim=1) # dim 1 because we need to simplify into a long vector for the simplicity of the algorythm or programming
            # The game of indexes like game of thrones.
        return index
        # Finally returning some modified version of index

# Let's create the model object in the main;
model=BigramLanguageModel(vocab_size).to(device) # Just putting the init variable to execute the construction, and it's pulling the nn materials from the superclass nn.Module
#m=model.to(device) # Just abreviating and sending the model wherre it belongs!
'''renaming to m because model is being used in the optimizer block'''
# Just sending our fine tuned indexes into the decode while unleashing the class methode generate where putting a zeros tensor as context.
#context=torch.zeros((1,1),dtype=torch.long,device=device) # Single input of zero; [o] as context and of course cuda!
'''This is our first token, token[0]=\n'''
# max_new_tokens is the kwarg for iterations in generate for loop, may be this is the bulk iteration phase.
#generated_chars=decode(m.generate(context,max_new_tokens=500)[0].tolist()) # Decode just takes in indexes, and our model determines how intelligent the INDEXES ARE!
# torch.int64 because it's a game of indexes!
'''
SO PROUD TO GENERATE THIS!
(3,0m-nRWbI,xA3kf)s6w2N_FTio80*hpa
6IACTFzUWSP2t8LhsFe 8hUR?R0*Z5CZa7jl6F;TI5[zURVbn3t.R_;NMOlCpirGD!;o(Fk 
5[UBmm.Mrb9_Cpp7OK9oY7Cp&Vw ,iPl!K)G n8ft81FHsnEiJ.iws4kH3ko1iB&ge(;X[D3T;Q_ADS4":G;Z55tmccH"YGN44"?t&?VG!aG,VBTj'?7I,-R9*c.:x!yEA
ggB?c'-]Pp!G)gbW:O447xOAup
5[oTSQpi14,seaA&!("8dcsFyRV&gNYS&_G"Tblv*nrT&VBoru:R,kG"eB1sJVovKIcRPJJ!ajgnxDt.]L9nf["l]LT4,tu"&!1VbSN?Ga!8C&to
R, "x DRPTR?YabtCFg3,Hl-Pl13k8:;I
q-CUdZ0wHdFr"r-LE&'':ok2(8K6tW"25tvOAH)gn?cQZsbwsTb9!_gKdcTY-Ey:8;7: tva"e,RVL3,TU_rj-VH)
'''
'''HAHAHAHAHAHAHAHA!'''
#generated_chars

# Now Optimizer
The so called `Training Phase`. We deal with train_data we don't do validation here.


1. loss.backward() — Calculating the Nablas (∇)

When logits, loss = model(xb, yb) finishes, loss is just a single number (e.g., 4.17). On its own, 4.17 doesn't tell you which weights in your embedding table caused the error.
When you call loss.backward():
1. PyTorch walks backward from loss through every grad_fn tag in your computation graph (using the calculus Chain Rule).
2. For every single number inside model.parameters() (like token_embedding_table.weight), it computes the partial derivative $\frac{\partial \text{Loss}}{\partial w}$—which you rightly called the Nabla (∇).
3. It stores those gradient numbers inside each parameter's .grad attribute (param.grad).
• Important: loss.backward() does NOT change any model weights yet. Your model's predictions are still 100% identical right after loss.backward() finishes. It only fills up param.grad with instructions saying: "If you increase this weight, the loss goes UP by this much; if you decrease it, the loss goes DOWN."

2. optimizer.step() — Updating the Weights

Once loss.backward() has filled param.grad for every weight, optimizer.step() reads those .grad values and actually modifies model.parameters() in place.
- In basic gradient descent (SGD), it simply subtracts a tiny fraction (lr, learning rate) of the gradient from each weight:
$$w_{\text{new}}=w_{\text{old}}-(\text{lr}\times w.\text{grad})$$
- Because you are using AdamW, it does a smarter version of that subtraction: it tracks momentum (moving averages of past gradients) and adapts the step size individually for every weight, then updates token_embedding_table.weight so that next time the model sees xb, its predicted logits will be closer to yb.
## 1. Mean Squared Error (MSE)

- Role: A loss function used primarily for regression (predicting continuous numbers like house prices or temperatures), unlike Cross-Entropy which is used for discrete classes/tokens.
$$\text{MSE}=\frac{1}{N}\sum _{i=1}^{N}(y_{i}-\hat{y}_{i})^{2}$$
(where $y_{i}$ is the true target value, $\hat{y}_{i}$ is the model's prediction, and N is the number of examples).
- Intuition: Squaring the error $(y_i - \hat{y}_i)^2$ ensures all errors are positive and penalizes large outliers heavily (an error of 4 gives a penalty of 16, whereas an error of 2 gives 4).

## 2. Gradient Descent (GD / SGD)

- Role: The core optimization algorithm that updates model weights (w) in the opposite direction of the gradient (∇ L, the steepest uphill slope) to minimize the loss.
$$g_{t}=\nabla _{w}L(w_{t})$$
$$w_{t+1}=w_{t}-\eta \cdot g_{t}$$
(where $g_{t}$ is the gradient/Nabla at step t, and η is the learning rate lr).
- Limitation: Uses the exact same step size η for every weight and can oscillate wildly back and forth in steep ravines or get stuck in flat saddle points.

## 3. Gradient Descent with Momentum `Tracks Direction`

- Role: Accelerates Gradient Descent by adding a "momentum" vector $m_{t}$ that remembers past gradients, just like a heavy ball rolling downhill builds speed in a consistent direction and dampens side-to-side zig-zagging, the so called accumulation.
$$m_{t}=\beta m_{t-1}+(1-\beta )g_{t}$$
$$w_{t+1}=w_{t}-\eta \cdot m_{t}$$
(where β ≈ 0.9 controls how much past momentum is kept, so subtler).
- Intuition: If gradients keep pointing in the same direction across steps, $m_{t}$ grows larger and moves faster; if gradients flip signs back and forth, they cancel each other out.

## 4. RMSprop (Root Mean Square Propagation)

- Role: `Adapts the learning rate individually` for every single parameter by dividing the step size by a running average of recent gradient magnitudes $v_{t}$.
$$v_{t}=\beta v_{t-1}+(1-\beta )g_{t}^{2}$$
$$w_{t+1}=w_{t}-\frac{\eta }{\sqrt{v_{t}}+\epsilon }\cdot g_{t}$$
(where $g_{t}^{2}$ is element-wise squared gradients, β ≈ 0.99, and ε ≈ 10⁻⁸ prevents division by zero).
- Intuition: Weights with huge, noisy gradients get divided by a large $\sqrt{v_{t}}$ (shrinking their effective learning rate so they don't explode), while rarely updated weights (like rare characters in nn.Embedding) get divided by a small $\sqrt{v_{t}}$ (boosting their learning rate). So subtle nudges on both sides of learning rate.

## 5. Adam (Adaptive Moment Estimation)

- Role: `Combines` Momentum (1st moment: mean of gradients $m_{t}$ and RMSprop (2nd moment: uncentered variance of gradients $v_{t}$, plus bias correction so the early steps don't start biased toward zero.
	1. Momentum (1st moment): $m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t$
	2. Adaptive Scale (2nd moment): $v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t^2$
	3. Bias Correction (jump start): $\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1 - \beta_2^t}$
	4. Weight Update: $w_{t+1} = w_t - \frac{\eta}{\sqrt{\hat{v}_t} + \epsilon} \cdot \hat{m}_t$

    (Default hyperparameters: β₁ = 0.9, β₂ = 0.999, ε = 10⁻⁸).

## 6. AdamW (Adam with Decoupled Weight Decay)

- Role: The standard optimizer used to train GPT, Transformers, and modern deep learning models. It fixes a subtle flaw in how standard Adam handles L2 regularization (weight decay).
- The Flaw in Standard Adam: Adding L2 penalty $(\frac{\lambda}{2} w^2)$ directly into the loss changes the gradient to $g_t + \lambda w_t$. Because Adam divides by $\sqrt{\hat{v}_{t}}$, `large weights with large gradients get their weight decay shrunk unintentionally`.
- The AdamW Fix (Decoupled Decay): Remove weight decay from $g_{t}$ completely as well, and instead shrink every weight directly by a small fraction $\eta \lambda w_t$ during the final update:
$$w_{t+1}=w_{t}-\eta \left(\frac{\hat{m}_{t}}{\sqrt{\hat{v}_{t}}+\epsilon }+\lambda w_{t}\right)$$
(where λ is the `weight_decay` hyperparameter).

# Purpose of `Weight Decay` $\eta \lambda w_t$
What Actually Happens Instead (Equilibrium, Not Stopping) is rather than ending training, weight decay creates a tug-of-war on every weight:
1. The gradient pulls w outward (away from 0) to fit the training data and lower the loss.
2. Weight decay pulls w inward (toward 0) proportionally to how big w currently is $(λ \times w)$.
Eventually, those two forces reach an equilibrium (balance point) where the outward pull of the data equals the inward pull of weight decay.
It prevents weights from growing infinitely large, while still keeping the model active and learning.

### `CHOOSING WHICH MODEL TO USE COMES DOWN TO PERFORMANCE, SPEED VS EFFICIENCY`
### `MAY BE WE NEED A SMALLER MODEL DOING FASTER TASKS, OR SELF ATTENTIVE MODELS`
## Learning Rate
- Higher learning rate is like taking long steps. Let's consider $f(x)=sin(x)$ where taking big steps at the steep $f'(x)$ descent might suddenly bring back into gradient ascend where cost is even higher, and it happens faster.
- Let's consider $\vec{\theta}'=\vec{\theta}-\alpha\vec{\nabla_{w,b}C}$ which involves a plethora of vectors with varying magnitudes and impacts as they contain both $\vec{b^l}$ biases and $W_{jk}^l$ weights where wights have varying costs depending on $\vec{a}_k^{l-1}$ and we end up with a non complanar surface representing the big $\vec{\theta}$ as a function.
- In this intricate situation subtle learning rate will give us most efficient $\vec{\theta}$.
The non coplanar surface can be a plane where biases are on x axis, weights on 2 axis and cost on Z axis. Then we will do partial gradients as snippets on bias axis or weights axis.
- AdamW is another layer of modification into Adam algoryth. It has weight decay, it generalizes the parameters more, the learning rate specially. It's the sweet spot in between high level and low level performance.
- The weight significance will actually shrink as it flattens out and make sure certain parameters in the network, of weight matrices isn't affcting the output of the model drastically in both positive or negative direction. There can be some lucky high performances and we need to rely on the solid rule of big numbers kinda things.

In [ ]:
# create a PyTorch optimizer, The So called 
'''The back propagation'''
optimizer=torch.optim.AdamW(model.parameters(),lr=learning_rate)
'''So we are doing the Nablas here. model.parameters() load all the nn parameters'''

for iter in range(max_iters): # global variable max_iter is how many iteration we are having in the training loop
    if iter%eval_iters==0:
        '''For this condition we are assigning estimate_loss after 250 iter here'''
        losses=estimate_loss()
        '''And while calling estimate_loss() we are running 500 iterations in that block under decorator @torch.no_grad()'''
        print(f"step: {iter}, train loss: {losses['train']:.3f}, val loss: {losses['val']:.3f}")
    # sample batch of data
    xb,yb=get_batch('train')
    '''We are sending 'train' flag to give us x,y batches from train_data'''
    # evaluate the loss;
    _,loss=model.forward(xb,yb)
    '''This came from the forward function i.e, loss calculation, final stage of feed forward'''
    '''It sends xb,yb as index and targets, so that means the first pass with context is done'''
    '''Now it's the second frorward pass'''
    '''We are sending first stacked batch with size=(4,8) tensor of chars[i] as index and second slided by one char same sized tensor of char indexes as targets.'''
    '''The one character sliding gives us the hat{y}, the labels to calculate loss!'''
    '''So now we have a 2D tensor as index, our very first batch from the train_data'''
    '''We successfully received new logits,loss from model.forward() only'''
    '''Still 2D tensor, our logits, and we did not use model.generate(index) yet'''
    optimizer.zero_grad(set_to_none=True) # It means we are also excluding 0. which takes place and set zer_grad to None instead of 0. float
    '''We just loaded an optimizer called, the so called nabla(w,b) zero_grad()'''
    '''By default pytocrh will accumulate the gradients over time, via adding them'''
    '''By using zero_grad() we prevent that accumulation from happening as we want fresh new gradients after each iteration'''
    '''Gradient accumulation is interesting though, it can be used for even better performing model'''
    loss.backward()
    '''It has vecor C and initiationg the the gradient descent on each delta s.'''
    '''Backward pass with our loss, it calculated all the nablas.'''
    '''It stores those gradient numbers inside each parameter's .grad attribute (param.grad).'''
    optimizer.step()
    '''Once loss.backward() has filled param.grad for every weight, optimizer.step() reads those .grad values and actually modifies model.parameters() in place.'''
    '''Updated and stored new weight matrixes and bias vectors into the model, so model got updated in one step'''
    '''It will keep repeating for max_iter times, and we need to analyze the optimizer so we all are on the same page.'''
    '''Probably after retweaking our model theta for max_iter times we will touch on the methods under generate()'''
    '''The more we run this our theta vector gets modifi'''
print(loss.item())
'''All the remaining variables are in the import cell'''
'''Still we have only one input and output layers'''
'''We see an array of losses from step: iiter=0, train loss: 2.576, val loss: 2.619 to step: 750, train loss: 2.567, val loss: 2.602'''

In [ ]:
context=torch.zeros((1,1),dtype=torch.long,device=device)
'''what happened to the other generation?'''
generated_chars=decode((model.generate(context,max_new_tokens=500))[0].tolist()) #.tolist() converts tensors into lists and it's taking the index[0] as there are 7 more per batch[i] as per bigram sliding process to incorporate parallel processing of cuda
'''context=0, starts with chars[0]=\n and max_new_tokens is just no ofiterations in generate() block'''
print(generated_chars)
# When we keep building on this, it will be a deepnet

## Some Activation Functions
1. ReLU(Rectified Unified Unit) Function:
   Applies rectified linear unit function element-wise:$$ReLU(x)^{+}=max(0,x)$$
   ReLU applies non-linearaity into our linear networks. So if our network is 100 layers deep, the activation will be applied to each of these layers and the network is going to learn a lot more things. It is going to learn a lot more non-linearly. It maps any input value between $0\ge y\ge\infty$.
- It prioritizes the bigger values, so more useful in weighted learning. In a computer we can never store absolute 0. floats so it never encounters hard zero errors.
#### This is exactly what the sofmax does, so more like `Sigmoids On Steroids!`
3. Sigmoid Function:
   Applies the element-wise function as follows:$$\sigma(x)=\frac{1}{1+e^{-x}}$$
   It looks extremerly non-linear, as non-linear as it can get. It maps any input value between $0\ge y\ge1$.
- Sigmoid function is more suitable for binary classification when either $y\rightarrow0$ or $y\rightarrow1$ and rounding the floats using y:.0f gives us boolean
4. TANH Function:
   Applies the Hyperbolic Tangent(Tanh) function element-wise).
   TanH is defined as:$$Tanh(x)=\frac{e^x-e^{-x}}{e^x+e^{-x}}=\frac{e^{2x}-1}{e^{2x}+1}$$
   By contrast It maps any input value between $-1\ge y\ge1$ and much steeper than sigmoid.
- For hidden layers in a neural network, especially for multi-layer perceptions, tanh is often more desirable because the mean of its output is closer to 0, which can make learning for the next layer easier.


In [ ]:
# We are gonna send them through some functions and simply test the outputs
x=torch.tensor([-0.05],dtype=torch.float32)
y=nn.ReLU(x) # For ReLU we don't need to use F
y=F.sigmoid(x) # It gives tensor([0.4875])
y=F.tanh(x) # Slower, it gives tensor([-0.0500])
y

# Transformer Architecture
#### `We are going to build it from scratch!`